<a href="https://colab.research.google.com/github/kalanakotawalagedara/Grid-box-identify/blob/main/Grid_Box_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **1) Identify pocket coordinates of protein based on co-crystalize ligand**


*   Type pdb ID
*   Obtain the center XYZ coordinates based on co-crystalized ligand


*   Not work for apo-proteins





In [13]:
# @title
import os, math, json, textwrap, re
from collections import defaultdict
from statistics import mean

# -------------------- User parameters --------------------
pdb_id = input("Enter PDB ID (e.g., 3ERT): ") or "3ERT"  # User input for PDB ID

# Validate PDB ID format
if not re.fullmatch(r'^[0-9][A-Za-z0-9]{3}$|^[A-Za-z0-9]{4}$', pdb_id):
    print(f"Error: Invalid PDB ID format '{pdb_id}'. PDB IDs typically consist of 4 alphanumeric characters (e.g., '3ERT').")
    download_success = False
    pdb_text = None
    master = {"pdb_id": pdb_id, "downloaded": download_success, "ligands": []}
else:
    output_dir = "/mnt/data"
    recommended_default_box = (30.0, 30.0, 30.0)  # default focused box (\u00C5)
    vina_exhaustiveness = 50
    contact_cutoff = 4.5  # \u00C5 for protein-ligand contacts
    # Common ignore list for glycans, solvents, and crystallization agents
    COMMON_IGNORE = {'HOH','WAT','NA','CL','K','MG','CA','SO4','PO4','EDO','GOL','DMS','MPD','ACE','NAG','SO3','ZN','MN','FE','NI','IOD','BMA', 'NDG', 'MAN', 'BOG'}

    # create output dir
    os.makedirs(output_dir, exist_ok=True)

    # -------------------- Parsing helpers --------------------
    def parse_pdb_text(pdb_text):
        protein_atoms = []  # list of dicts {'chain','resseq','resname','atom','x','y','z'}
        het_atoms_by_res = defaultdict(list)  # key: (resname, chain, resseq) -> list of atom dicts
        for line in pdb_text.splitlines():
            if len(line) < 54:
                continue
            record = line[0:6].strip()
            if record in ("ATOM","HETATM"):
                atom_name = line[12:16].strip()
                resname = line[17:20].strip()
                chain = line[21].strip() or "_"
                resseq = line[22:26].strip()
                try:
                    x = float(line[30:38].strip())
                    y = float(line[38:46].strip())
                    z = float(line[46:54].strip())
                except Exception:
                    continue
                entry = {"atom": atom_name, "resname": resname, "chain": chain, "resseq": resseq, "x": x, "y": y, "z": z}
                if record == "ATOM":
                    protein_atoms.append(entry)
                else:
                    het_atoms_by_res[(resname, chain, resseq)].append(entry)
        return protein_atoms, het_atoms_by_res

    def compute_centroid(atom_list):
        xs = [a['x'] for a in atom_list]
        ys = [a['y'] for a in atom_list]
        zs = [a['z'] for a in atom_list]
        return (mean(xs), mean(ys), mean(zs))

    def ligand_extent(atom_list):
        xs = [a['x'] for a in atom_list]
        ys = [a['y'] for a in atom_list]
        zs = [a['z'] for a in atom_list]
        extent_x = max(xs)-min(xs) if xs else 0.0
        extent_y = max(ys)-min(ys) if ys else 0.0
        extent_z = max(zs)-min(zs) if zs else 0.0
        return extent_x, extent_y, extent_z

    def distance(a,b):
        return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2 + (a[2]-b[2])**2)

    def ligand_protein_contacts(lig_atoms, prot_atoms, cutoff=4.5):
        contacts = set()
        contact_list = []
        for la in lig_atoms:
            lcoord = (la['x'], la['y'], la['z'])
            for pa in prot_atoms:
                pcoord = (pa['x'], pa['y'], pa['z'])
                if distance(lcoord, pcoord) <= cutoff:
                    contacts.add((pa['resname'], pa['chain'], pa['resseq']))
                    contact_list.append(pa)
        return contacts, contact_list

hydrophobic_residues = {"ALA","VAL","ILE","LEU","PHE","TRP","TYR","MET"}

def classify_site(contacts, contact_atoms):
    num_contacts = len(contacts)
    if contact_atoms:
        resnames = [a['resname'] for a in contact_atoms]
        hydrophobic_count = sum(1 for r in resnames if r.upper() in hydrophobic_residues)
        hydrophobic_fraction = hydrophobic_count / len(resnames)
    else:
        hydrophobic_fraction = 0.0
    if num_contacts >= 8 and hydrophobic_fraction >= 0.4:
        classification = "Likely orthosteric (buried in hydrophobic pocket)"
    elif num_contacts >= 8:
        classification = "Likely orthosteric (buried)"
    elif 3 <= num_contacts <= 7:
        classification = "Ambiguous (possible allosteric or shallow orthosteric)"
    elif num_contacts <= 2 and num_contacts > 0:
        classification = "Likely allosteric or peripheral (few contacts)"
    else:
        classification = "No protein contacts detected"
    return classification, num_contacts, hydrophobic_fraction

# -------------------- Fetch PDB --------------------
pdb_text = None
download_success = False
try:
    import requests
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    pdb_text = r.text
    download_success = True
except Exception as e:
    print(f"Warning: could not download PDB {pdb_id}.")
    pdb_text = None

# -------------------- Main processing --------------------
master = {"pdb_id": pdb_id, "downloaded": download_success, "ligands": []}

if pdb_text is not None:
    protein_atoms, het_atoms_by_res = parse_pdb_text(pdb_text)
    # Filter to only relevant ligands by checking against the ignore list
    ligand_groups = {k:v for k,v in het_atoms_by_res.items() if k[0].upper() not in COMMON_IGNORE}

    if not ligand_groups:
        print("No relevant ligands found in this PDB file.")

    for (resname, chain, resseq), atoms in ligand_groups.items():
        centroid = compute_centroid(atoms)
        ext_x, ext_y, ext_z = ligand_extent(atoms)
        padding = 8.0
        suggested_size_x = max(recommended_default_box[0], ext_x + padding*2)
        suggested_size_y = max(recommended_default_box[1], ext_y + padding*2)
        suggested_size_z = max(recommended_default_box[2], ext_z + padding*2)
        suggested_size = (round(suggested_size_x,3), round(suggested_size_y,3), round(suggested_size_z,3))
        contacts, contact_atoms = ligand_protein_contacts(atoms, protein_atoms, cutoff=contact_cutoff)
        classification, num_contacts, hydrophobic_fraction = classify_site(contacts, contact_atoms)
        rec = {
            "resname": resname, "chain": chain, "resseq": resseq,
            "centroid": {"x": round(centroid[0],3), "y": round(centroid[1],3), "z": round(centroid[2],3)},
            "ligand_atom_count": len(atoms),
            "extent": {"x": round(ext_x,3), "y": round(ext_y,3), "z": round(ext_z,3)},
            "recommended_box": {"size_x": suggested_size[0], "size_y": suggested_size[1], "size_z": suggested_size[2]},
            "classification": classification,
            "num_contacts": num_contacts,
            "hydrophobic_fraction": round(hydrophobic_fraction,3)
        }
        master["ligands"].append(rec)

# -------------------- Write outputs --------------------
master_txt_lines = [f"# Master docking grid report for PDB: {pdb_id}", f"# Downloaded: {download_success}", ""]
for lig in master["ligands"]:
    master_txt_lines.append(f"--- Ligand: {lig['resname']} (chain {lig['chain']}, resseq {lig['resseq']}) ---")
    master_txt_lines.append(f"Binding site center (\u00C5): {lig['centroid']['x']}, {lig['centroid']['y']}, {lig['centroid']['z']}")
    master_txt_lines.append(f"Search space size (\u00C5): {lig['recommended_box']['size_x']}, {lig['recommended_box']['size_y']}, {lig['recommended_box']['size_z']}")
    master_txt_lines.append(f"Contacts: {lig['num_contacts']}")
    master_txt_lines.append(f"Site: {lig['classification']}")
    master_txt_lines.append("")

master_txt = "\n".join(master_txt_lines)
master_txt_path = os.path.join(output_dir, f"master_report_{pdb_id}.txt")
master_json_path = os.path.join(output_dir, f"master_report_{pdb_id}.json")
with open(master_txt_path, "w") as f: f.write(master_txt)
with open(master_json_path, "w") as f: json.dump(master, f, indent=2)
created_files = [master_txt_path, master_json_path]
for lig in master["ligands"]:
    safe_label = f"{pdb_id}_{lig['resname']}_{lig['chain']}_{lig['resseq']}"
    vina_conf = textwrap.dedent(f"""
    center_x = {lig['centroid']['x']}
    center_y = {lig['centroid']['y']}
    center_z = {lig['centroid']['z']}
    size_x = {lig['recommended_box']['size_x']}
    size_y = {lig['recommended_box']['size_y']}
    size_z = {lig['recommended_box']['size_z']}
    exhaustiveness = {vina_exhaustiveness}
    """).strip()
    vina_conf_path = os.path.join(output_dir, f"vina_conf_{safe_label}.conf")
    with open(vina_conf_path, "w") as f: f.write(vina_conf)
    created_files.append(vina_conf_path)

Enter PDB ID (e.g., 3ERT): 3N8W


In [14]:
# @title
import pandas as pd

# Prepare data for CSV
csv_data = []
for lig in master['ligands']:
    csv_data.append({
        'pdb_id': master['pdb_id'],
        'resname': lig['resname'],
        'chain': lig['chain'],
        'resseq': lig['resseq'],
        'center_x': lig['centroid']['x'],
        'center_y': lig['centroid']['y'],
        'center_z': lig['centroid']['z'],
        'size_x': lig['recommended_box']['size_x'],
        'size_y': lig['recommended_box']['size_y'],
        'size_z': lig['recommended_box']['size_z']
    })

# Create DataFrame
ligand_coords_df = pd.DataFrame(csv_data)

# Define the output path for the CSV file
csv_output_path = os.path.join(output_dir, f"ligand_grid_coordinates_{pdb_id}.csv")

# Save to CSV
ligand_coords_df.to_csv(csv_output_path, index=False)

print(f"CSV file created: {csv_output_path}")

# Display the DataFrame head
display(ligand_coords_df.head())

# Add to created files list for download links
created_files.append(csv_output_path)

CSV file created: /mnt/data/ligand_grid_coordinates_3N8W.csv


,pdb_id,resname,chain,resseq,center_x,center_y,center_z,size_x,size_y,size_z
0,3N8W,HEM,A,801,41.658,-61.029,-2.684,30.0,30.0,30.0
1,3N8W,FLP,A,802,32.297,-44.194,1.154,30.0,30.0,30.0
2,3N8W,HEM,B,601,30.418,-67.221,47.343,30.0,30.0,30.0


# **2) Identify pocket coordinates from apo-protein**


*   Install Fpocket
*   Upload .pdb file (better to use Apo-protein)



In [ ]:
# @title
# Install condacolab (if not already installed)
!pip install -q condacolab
import condacolab
condacolab.install()

# After condacolab installation and kernel restart, install fpocket
print("\n--- Installing Fpocket via Conda ---")
# Use 'bash -c' to ensure commands are executed in a shell context where conda/mamba is initialized
install_fpocket_command = """
source /usr/local/etc/profile.d/conda.sh
conda activate base
conda install -c conda-forge -c bioconda fpocket -y
"""

try:
    subprocess.run(install_fpocket_command, shell=True, capture_output=True, text=True, check=True, executable="/bin/bash")
    print("Fpocket and Conda environment setup complete.")
except subprocess.CalledProcessError as e:
    print(f"Error installing Fpocket: {e}")
    print(f"Stdout: {e.stdout}\nStderr: {e.stderr}")
    sys.exit(1)

# Verify installation
check_fpocket = subprocess.run("which fpocket", shell=True, capture_output=True, text=True, executable="/bin/bash")
if check_fpocket.returncode == 0:
    print(f"Fpocket executable found at: {check_fpocket.stdout.strip()}")
else:
    print("Warning: Fpocket command not found. Please ensure installation was successful.")

print("\n--- Setup complete. You can now run the next cell to upload your PDB and detect pockets. ---")


In [ ]:
# @title Fpocket Pocket Detection
import os, re, shutil, subprocess, sys
import pandas as pd
from google.colab import files

print("--- Fpocket Pocket Detection ---")

# ── 1. Upload PDB ────────────────────────────────────────────────────
print("\nUpload your PDB file:")
uploaded = files.upload()
if not uploaded:
    sys.exit("No file uploaded.")

original_name = list(uploaded.keys())[0]
print(f"Uploaded: {original_name}")

safe_name = re.sub(r'[\s()[\].]+', '_', original_name).strip('_') # Updated regex to handle '.' better
if '.' in safe_name:
    parts = safe_name.rsplit('.', 1)
    safe_name = parts[0] + '.pdb' if parts[1].lower() != 'pdb' else safe_name
else:
    safe_name += '.pdb'

if safe_name != original_name:
    shutil.copy(original_name, safe_name)
    print(f"Renamed: '{original_name}' → '{safe_name}'")

base    = safe_name.rsplit('.', 1)[0]
out_dir = f"{base}_out"

# ── 2. Run fpocket ───────────────────────────────────────────────────
print(f"\nRunning fpocket on '{safe_name}'...")
run = subprocess.run(
    f'source /usr/local/etc/profile.d/conda.sh && conda activate base && fpocket -f "{safe_name}"',
    shell=True, capture_output=True, text=True, executable="/bin/bash"
)
if run.returncode != 0:
    print("FPOCKET FAILED")
    print("stdout:", run.stdout)
    print("stderr:", run.stderr)
    sys.exit(1)
print("Fpocket completed successfully.")

# ── 3. Locate output directory ───────────────────────────────────────
if not os.path.isdir(out_dir):
    candidates = [d for d in os.listdir('.') if d.endswith('_out') and os.path.isdir(d)]
    out_dir = candidates[0] if candidates else None
if not out_dir:
    print("ERROR: output dir not found. Dirs:", [d for d in os.listdir('.') if os.path.isdir(d)])
    sys.exit(1)

# ── 4. Locate _info.txt ──────────────────────────────────────────────
info_path = os.path.join(out_dir, f"{base}_info.txt")
if not os.path.exists(info_path):
    hits = [f for f in os.listdir(out_dir) if f.endswith('_info.txt')]
    info_path = os.path.join(out_dir, hits[0]) if hits else None
if not info_path or not os.path.exists(info_path):
    print("ERROR: _info.txt not found. Files:", os.listdir(out_dir))
    sys.exit(1)

# ── 5. Debug: print raw file so format is visible ────────────────────
# The user explicitly asked to remove this debug output.
# print(f"\n--- Raw _info.txt (first 3000 chars) ---")
with open(info_path) as f:
    raw = f.read()
# print(raw[:3000])
# print("--- End raw ---\n")

# ── 6. Parse _info.txt (handles all known fpocket output variants) ────
def save_pocket(p):
    p.pop('_awaiting_coords', None)
    p.pop('_coord_count',     None)
    return p

pockets_data  = []
current_pocket = {}

for line in raw.splitlines():
    line = line.strip()

    # ── New pocket block ─────────────────────────────────────────────
    m = re.match(r'POCKET\s+(\d+)\s*:', line, re.IGNORECASE)
    if m:
        if current_pocket:
            pockets_data.append(save_pocket(current_pocket))
        current_pocket = {
            'Pocket ID':           int(m.group(1)),
            'Druggability Score':  None,
            'Volume (Å³)':         None,
            'Center X':            None,
            'Center Y':            None,
            'Center Z':            None,
        }
        continue

    if not current_pocket:
        continue

    # ── Druggability Score ───────────────────────────────────────────
    ds = re.search(r'Druggability Score\s*:\s*([-+]?\d*\.?\d+(?:[eE][+-]?\d+)?)', line, re.IGNORECASE)
    if ds:
        current_pocket['Druggability Score'] = float(ds.group(1))
        continue

    # ── Volume ───────────────────────────────────────────────────────
    vol = re.search(r'Volume\s*:\s*([-+]?\d*\.?\d+(?:[eE][+-]?\d+)?)', line, re.IGNORECASE)
    if vol:
        current_pocket['Volume (Å³)'] = float(vol.group(1))
        continue

    # ── Pocket center (all variants) ─────────────────────────────────
    # Variant A: "Pocket center    :  x.xx  y.yy  z.zz"  (v3, all on one line)
    # Variant B: "Pocket center (x, y, z) :  x.xx  y.yy  z.zz"
    ctr = re.search(r'[Pp]ocket\s+[Cc]enter.*?::\s*(.*)', line) # Fix: make sure it's ':' not '::'
    if ctr:
        coords = re.findall(r'[-+]?\d*\.?\d+(?:[eE][+-]?\d+)?', ctr.group(1))
        if len(coords) >= 3:
            current_pocket['Center X'] = float(coords[0])
            current_pocket['Center Y'] = float(coords[1])
            current_pocket['Center Z'] = float(coords[2])
        else:
            # Coords will follow on next lines — flag it
            current_pocket['_awaiting_coords'] = True
            current_pocket['_coord_count']     = 0
        continue

    # Variant C: coordinates arrive on separate lines after center header
    if current_pocket.get('_awaiting_coords'):
        nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][+-]?\d+)?', line)
        if nums:
            # Ensure there are enough numbers to assign to X, Y, Z sequentially
            if current_pocket['_coord_count'] == 0 and len(nums) >= 1:
                current_pocket['Center X'] = float(nums.pop(0))
                current_pocket['_coord_count'] += 1
            if current_pocket['_coord_count'] == 1 and len(nums) >= 1:
                current_pocket['Center Y'] = float(nums.pop(0))
                current_pocket['_coord_count'] += 1
            if current_pocket['_coord_count'] == 2 and len(nums) >= 1:
                current_pocket['Center Z'] = float(nums.pop(0))
                current_pocket['_coord_count'] += 1

            if current_pocket['_coord_count'] >= 3:
                current_pocket.pop('_awaiting_coords')
                current_pocket.pop('_coord_count')
        continue

    # Variant D: explicit Cx / Cy / Cz labels (fpocket v2)
    for axis, key in [('x', 'Center X'), ('y', 'Center Y'), ('z', 'Center Z')]:
        m2 = re.search(rf'\bC{axis}\s*:\s*([-+]?\d*\.?\d+(?:[eE][+-]?\d+)?)', line, re.IGNORECASE)
        if m2:
            current_pocket[key] = float(m2.group(1))

# Save last pocket
if current_pocket and 'Pocket ID' in current_pocket:
    pockets_data.append(save_pocket(current_pocket))

# Fallback for center coordinates: calculate from PQR if not found in info.txt
for pocket in pockets_data:
    if pocket['Center X'] is None or pocket['Center Y'] is None or pocket['Center Z'] is None:
        pocket_id = pocket['Pocket ID']
        # Fpocket stores pqr files in a 'pockets' subdirectory within out_dir
        pqr_path = os.path.join(out_dir, 'pockets', f'pocket{pocket_id}_vert.pqr')

        if os.path.exists(pqr_path):
            x_coords = []
            y_coords = []
            z_coords = []
            with open(pqr_path, 'r') as pqr_f:
                for line_pqr in pqr_f:
                    # PQR files typically have ATOM or HETATM records for vertices
                    if line_pqr.startswith('ATOM') or line_pqr.startswith('HETATM'):
                        try:
                            # Standard PDB/PQR coordinate fields
                            x = float(line_pqr[30:38].strip())
                            y = float(line_pqr[38:46].strip())
                            z = float(line_pqr[46:54].strip())
                            x_coords.append(x)
                            y_coords.append(y)
                            z_coords.append(z)
                        except ValueError:
                            # Skip lines that don't have valid float coordinates
                            pass
            if x_coords: # If coordinates were found, compute centroid
                pocket['Center X'] = round(sum(x_coords) / len(x_coords), 3)
                pocket['Center Y'] = round(sum(y_coords) / len(y_coords), 3)
                pocket['Center Z'] = round(sum(z_coords) / len(z_coords), 3)
            # else: keep as None if PQR exists but no valid coords were found
        # else: keep as None if PQR file not found for this pocket


# ── 7. Display and export ────────────────────────────────────────────
if not pockets_data:
    print("⚠ No pockets parsed. Check the raw output above to see the exact format fpocket used.")
else:
    df = pd.DataFrame(pockets_data)
    if 'Druggability Score' in df.columns:
        df = df.sort_values('Druggability Score', ascending=False).reset_index(drop=True)

    print(f"✓ {len(df)} pocket(s) found — ranked by Druggability Score:\n")
    display(df)

    csv_path = os.path.join(out_dir, f"fpocket_pockets_{base}.csv")
    df.to_csv(csv_path, index=False)
    print(f"\nSaved: {csv_path}")
    files.download(csv_path)


# **3) Cleaning-up for space**

In [ ]:
# @title
import os
import shutil

print("Cleaning up /content except for sample_data...")

# Get a list of all items in /content
items_in_content = os.listdir('/content')

for item in items_in_content:
    # Skip 'sample_data' directory
    if item == 'sample_data':
        continue

    item_path = os.path.join('/content', item)

    try:
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
            print(f"  Removed directory: {item}")
        elif os.path.isfile(item_path):
            os.remove(item_path)
            print(f"  Removed file: {item}")
    except OSError as e:
        print(f"Error removing {item_path}: {e}")

print("Cleanup complete. Only 'sample_data' should remain.")

# Optionally, verify the contents after cleanup
# !ls -R /content

Cleaning up /content except for sample_data...
Cleanup complete. Only 'sample_data' should remain.
